![Banner Deputados](https://www.camara.leg.br/internet/deputado/img/logo-camara.png)

# 🏛️ Fast Track — 01. Bronze: Deputados

**Ingestão de Dados Brutos — API Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir dados brutos de deputados da API da Câmara dos Deputados para a camada Bronze |
| **Entidade** | Deputados (informações básicas, histórico de mandatos) |
| **Endpoint API** | `https://dadosabertos.camara.leg.br/api/v2/deputados` |
| **Tabela Destino** | `uc_fast_track.bronze.deputados_raw` |
| **Modo** | APPEND (histórico completo de todas as execuções) |
| **Checkpoint** | Por `id` do deputado (último ID processado) |
| **Formato Bronze** | Payload JSON raw (STRING) + colunas técnicas |
| **Responsável** | Ernesto Bassoli |
| **Programa** | Upskill Tiller – Engenharia de Dados (T2.7) |

---

## 📋 O Que Este Notebook Faz

Este notebook realiza a **primeira camada de ingestão** (Bronze) do pipeline Fast Track:

1. **Recupera o Load ID** gerado pelo notebook 00_init_run
2. **Busca o último checkpoint** (último ID de deputado processado)
3. **Chama a API** da Câmara dos Deputados com paginação automática
4. **Salva o payload JSON raw** em `bronze.deputados_raw` com colunas técnicas
5. **Registra logs detalhados** em `log.bronze_ingest_runs` e `ops.ingestion_requests`
6. **Atualiza checkpoint** para próxima execução incremental
7. **Valida** os dados ingeridos (contagem, amostra)

**IMPORTANTE:** Este notebook mantém o payload JSON COMPLETO sem transformações. Parse e limpeza acontecem na camada Silver.

---

## 🔄 Quando Este Notebook É Executado

* **Workflow A (Diário)**: Task A01_bronze_deputados
* **Workflow B (Micro-batch 10 min)**: Task B01_bronze_deputados
* **Manualmente**: Para reprocessamento ou testes

---

## 🎯 Entregas Deste Notebook

Ao final da execução:

✅ **Dados Ingeridos**: N registros novos em `bronze.deputados_raw`
✅ **Checkpoint Atualizado**: Último ID processado salvo em `ops.checkpoints`
✅ **Logs Registrados**: 1 linha em `log.bronze_ingest_runs` com métricas
✅ **Requests Logados**: N linhas em `ops.ingestion_requests` (1 por chamada HTTP)
✅ **Validação**: Contagem e amostra dos dados exibidos

## ⚙️ Bloco 1: Configuração e Recuperação do Load ID

### 📋 Descrição

Este bloco recupera as **variáveis globais e o Load ID** criados pelo notebook `00_init_run`.

---

### 🔗 Dependência do Notebook 00

Este notebook **depende** do notebook `00_init_run` ter sido executado primeiro, pois precisa de:

* **`load_id`**: UUID único que identifica esta execução
* **`env`**: Ambiente (dev/hml/prd)
* **`run_date`**: Data de referência desta execução
* **`full_refresh`**: Se deve reprocessar tudo ou apenas incremental
* **`catalog`**: Nome do catálogo UC (`uc_fast_track`)
* **`schema_bronze`, `schema_ops`, `schema_log`**: Nomes dos schemas

---

### 🔍 Como Funciona

Em Databricks, quando um notebook é executado via workflow:
1. O notebook **00_init_run** (task anterior) executa e define variáveis
2. Este notebook **herda automaticamente** essas variáveis do contexto
3. Usamos `%run` ou acessamos diretamente as variáveis já disponíveis

**IMPORTANTE:** Se executar este notebook **manualmente** (fora do workflow), você deve executar o notebook 00 primeiro ou definir as variáveis manualmente.

---

### 💡 Para Leigos

Pense no notebook 00 como a **recepção de um hotel** que entrega a chave do quarto (load_id).
Este notebook é o **hóspede** que precisa da chave para entrar no quarto.
Sem passar pela recepção primeiro, não consegue acessar o quarto!

In [0]:
# ============================================================
# RECUPERAÇÃO DE VARIÁVEIS DO NOTEBOOK 00_init_run
# ============================================================
# Este notebook depende das variáveis definidas no notebook 00.
# Se executar manualmente, execute o notebook 00 primeiro.
# ============================================================

print("🔗 Recuperando variáveis do notebook 00_init_run...")
print()

# ============================================================
# VERIFICAÇÃO: Load ID Disponível?
# ============================================================
# O load_id é a variável mais crítica - identifica esta execução.

try:
    # Tenta acessar o load_id definido no notebook 00
    print(f"✅ Load ID encontrado: {load_id}")
    print(f"   Este UUID identifica esta execução específica")
except NameError:
    # Se não encontrar, o notebook 00 não foi executado
    print("❌ ERRO: Load ID não encontrado!")
    print("   Execute o notebook 00_init_run primeiro")
    raise Exception("Load ID não disponível - execute notebook 00 primeiro")

print()

# ============================================================
# RECUPERAÇÃO: Outras Variáveis Globais
# ============================================================
# Recupera env, run_date, full_refresh, catalog, schemas.

try:
    print(f"✅ Ambiente (env):              {env}")
    print(f"✅ Data de Execução (run_date): {run_date}")
    print(f"✅ Full Refresh:                {full_refresh}")
    print()
    print(f"✅ Catálogo UC:                 {catalog}")
    print(f"✅ Schema Bronze:               {schema_bronze}")
    print(f"✅ Schema Ops:                  {schema_ops}")
    print(f"✅ Schema Log:                  {schema_log}")
    print()
except NameError as e:
    print(f"❌ ERRO: Variável não encontrada: {e}")
    print("   Execute o notebook 00_init_run primeiro")
    raise

# ============================================================
# DEFINIÇÃO: Variáveis Específicas deste Notebook
# ============================================================
# Define constantes específicas para ingestão de deputados.

# Nome da entidade (usado em logs e tabelas)
entity_name = "deputados"

# Endpoint da API
api_base_url = "https://dadosabertos.camara.leg.br/api/v2"
api_endpoint = f"{api_base_url}/deputados"

# Nome da tabela Bronze de destino
table_bronze = f"{catalog}.{schema_bronze}.{entity_name}_raw"

# Configurações de paginação
page_size = 100  # Registros por página (máximo da API)
max_retries = 3  # Tentativas em caso de erro HTTP
retry_delay_seconds = 5  # Pausa entre tentativas

print("✅ Variáveis específicas definidas:")
print(f"   • Entidade:           {entity_name}")
print(f"   • Endpoint API:       {api_endpoint}")
print(f"   • Tabela Destino:     {table_bronze}")
print(f"   • Page Size:          {page_size}")
print()
print("="*70)

## 📦 Bloco 2: Importações Específicas

### 📋 Descrição

Importações adicionais específicas para este notebook (além das já carregadas no notebook 00).

---

### 📚 Bibliotecas Importadas

* **`requests`**: Para chamadas HTTP à API
* **`json`**: Para manipular payloads JSON
* **`time`**: Para pausas entre requisições (rate limiting)
* **`uuid`, `hashlib`**: Para gerar IDs e hashes
* **PySpark**: Para manipular DataFrames e salvar em Delta

A maioria já foi importada no notebook 00, mas declaramos novamente por segurança (imports são idempotentes).

In [0]:
# ============================================================
# IMPORTAÇÕES ESPECÍFICAS PARA INGESTÃO BRONZE
# ============================================================
# Importa bibliotecas necessárias (maioria já foi importada no notebook 00).
# ============================================================

print("📦 Verificando bibliotecas...")

# Bibliotecas Python nativas
import requests
import json
import time
import uuid
import hashlib
from datetime import datetime

# PySpark
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
from pyspark.sql.functions import col, lit, current_timestamp, sha2

print("✅ Bibliotecas disponíveis")
print()

## 🔌 Bloco 3: Função de Extração da API

### 📋 Descrição

Este bloco define a função `get_deputados_from_api()` que:
1. Chama o endpoint `/deputados` da API
2. Trata paginação automática (múltiplas páginas)
3. Implementa retry com backoff exponencial
4. Loga cada request HTTP em `ops.ingestion_requests`
5. Retorna lista de payloads JSON

---

### 🌐 Endpoint da API

**URL Base:** `https://dadosabertos.camara.leg.br/api/v2/deputados`

**Parâmetros:**
* `pagina`: Número da página (começa em 1)
* `itens`: Registros por página (máx 100)
* `ordem`: Ordenação (ASC/DESC)
* `ordenarPor`: Campo para ordenar (ex: `id`)

**Resposta:**
```json
{
  "dados": [
    {"id": 123, "nome": "João Silva", "siglaPartido": "PT", ...},
    {"id": 124, "nome": "Maria Santos", "siglaPartido": "PSDB", ...}
  ],
  "links": [
    {"rel": "next", "href": "...pagina=2"},
    {"rel": "last", "href": "...pagina=15"}
  ]
}
```

---

### 🔄 Lógica de Paginação

1. Começa na página 1
2. Chama API e pega `dados`
3. Verifica se existe `links.next`
4. Se sim, incrementa página e repete
5. Se não, termina (última página)

---

### 💡 Para Leigos

Imagine que você está lendo um livro muito grande:
* **Página 1**: Primeiros 100 deputados
* **Página 2**: Próximos 100 deputados
* **Página N**: Últimos deputados

A API não retorna todos de uma vez (seria muito pesado). Ela divide em "páginas" de 100 registros. Nossa função automaticamente vai virando as páginas até chegar no final!

In [0]:
# ============================================================
# FUNÇÃO: EXTRAÇÃO DE DEPUTADOS DA API
# ============================================================
# Chama a API, trata paginação, implementa retry, loga requests.
# ============================================================

def get_deputados_from_api(checkpoint_id=None):
    """
    Extrai lista de deputados da API da Câmara dos Deputados.
    
    Args:
        checkpoint_id (int, optional): Último ID processado (para incremental).
                                        Se None, processa todos.
    
    Returns:
        list: Lista de dicts com payloads JSON dos deputados.
    """
    
    print(f"🔌 Iniciando extração da API: {api_endpoint}")
    print()
    
    # Lista para acumular todos os deputados de todas as páginas
    all_deputados = []
    
    # Controle de paginação
    pagina_atual = 1
    has_more_pages = True
    total_requests = 0
    total_records = 0
    
    # ============================================================
    # LOOP DE PAGINAÇÃO
    # ============================================================
    # Continua até não haver mais páginas.
    
    while has_more_pages:
        
        print(f"📄 Processando página {pagina_atual}...")
        
        # ============================================================
        # PASSO 1: Montar Parâmetros da Requisição
        # ============================================================
        
        params = {
            "pagina": pagina_atual,
            "itens": page_size,
            "ordem": "ASC",
            "ordenarPor": "id"
        }
        
        # Se há checkpoint, filtra apenas IDs maiores que o último processado
        # (a API não suporta filtro direto, então filtraremos após receber)
        
        # ============================================================
        # PASSO 2: Fazer Request HTTP com Retry
        # ============================================================
        
        request_id = str(uuid.uuid4())  # ID único deste request
        request_start_time = time.time()
        
        success = False
        response_data = None
        http_status = None
        error_message = None
        
        for attempt in range(1, max_retries + 1):
            try:
                # Faz o request HTTP GET
                response = requests.get(
                    api_endpoint,
                    params=params,
                    timeout=30
                )
                
                http_status = response.status_code
                
                # Se sucesso (200), processa resposta
                if response.status_code == 200:
                    response_data = response.json()
                    success = True
                    break
                
                # Se erro, loga e tenta novamente
                else:
                    error_message = f"HTTP {response.status_code}"
                    print(f"   ⚠️  Tentativa {attempt}/{max_retries} falhou: {error_message}")
                    
                    if attempt < max_retries:
                        time.sleep(retry_delay_seconds * attempt)  # Backoff exponencial
            
            except requests.exceptions.RequestException as e:
                error_message = str(e)
                print(f"   ⚠️  Tentativa {attempt}/{max_retries} falhou: {error_message}")
                
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds * attempt)
        
        # ============================================================
        # PASSO 3: Calcular Tempo de Resposta e Logar Request
        # ============================================================
        
        response_time_ms = int((time.time() - request_start_time) * 1000)
        total_requests += 1
        
        # Cria DataFrame com log deste request HTTP
        log_request_data = [
            {
                "request_id": request_id,
                "load_id": load_id,
                "entity_name": entity_name,
                "endpoint": f"{api_endpoint}?pagina={pagina_atual}",
                "http_method": "GET",
                "http_status": http_status,
                "response_time_ms": response_time_ms,
                "success": success,
                "error_message": error_message,
                "request_timestamp": datetime.now()
            }
        ]
        
        df_log_request = spark.createDataFrame(log_request_data)
        
        # Salva log em ops.ingestion_requests (APPEND)
        df_log_request.write.mode("append").saveAsTable(f"{catalog}.{schema_ops}.ingestion_requests")
        
        # ============================================================
        # PASSO 4: Verificar Se Request Foi Bem-Sucedido
        # ============================================================
        
        if not success:
            print(f"   ❌ Falha após {max_retries} tentativas. Abortando.")
            raise Exception(f"Falha ao chamar API após {max_retries} tentativas: {error_message}")
        
        # ============================================================
        # PASSO 5: Processar Resposta (Extrair 'dados')
        # ============================================================
        
        deputados_page = response_data.get("dados", [])
        records_in_page = len(deputados_page)
        
        print(f"   ✅ {records_in_page} registros recebidos ({response_time_ms}ms)")
        
        # Filtrar por checkpoint (se incremental)
        if checkpoint_id is not None:
            deputados_page = [d for d in deputados_page if d.get("id", 0) > checkpoint_id]
            print(f"   🔍 Filtrados por checkpoint (id > {checkpoint_id}): {len(deputados_page)} novos")
        
        # Adiciona à lista acumulada
        all_deputados.extend(deputados_page)
        total_records += len(deputados_page)
        
        # ============================================================
        # PASSO 6: Verificar Se Há Próxima Página
        # ============================================================
        
        links = response_data.get("links", [])
        has_next = any(link.get("rel") == "next" for link in links)
        
        if has_next and len(deputados_page) > 0:
            pagina_atual += 1
            time.sleep(0.5)  # Pausa educada entre requests (rate limiting)
        else:
            has_more_pages = False
            print(f"   ℹ️  Última página alcançada")
        
        print()
    
    # ============================================================
    # RESULTADO FINAL
    # ============================================================
    
    print("="*70)
    print(f"✅ Extração concluída")
    print(f"   • Total de requests HTTP: {total_requests}")
    print(f"   • Total de registros extraídos: {total_records}")
    print("="*70)
    print()
    
    return all_deputados

# Confirmação de função carregada
print("✅ Função get_deputados_from_api() definida")
print()

## 🔖 Bloco 4: Checkpoint e Modo Incremental

### 📋 Descrição

Este bloco busca o **último checkpoint** salvo para determinar de onde continuar o processamento.

---

### 🎯 O Que é Checkpoint?

O checkpoint é um **marcador** que indica até onde o pipeline já processou.

**Para deputados, usamos:** `checkpoint_type = 'last_id'` → Último ID de deputado processado

**Exemplo:**
* **Execução 1**: Processa deputados ID 1-500 → Checkpoint = 500
* **Execução 2**: Começa do ID 501 (incremental)
* **Execução 3**: Processa apenas novos (ex: 501-520)

---

### 🔄 Full Refresh vs Incremental

**Full Refresh (`full_refresh=true`):**
* Ignora checkpoint
* Reprocessa TUDO do zero
* Usado em: primeira execução, correção de erros, mudança de lógica

**Incremental (`full_refresh=false`):**
* Usa checkpoint
* Processa apenas o que é novo
* Modo padrão para execuções diárias/micro-batch

---

### 📊 Tabela: `ops.checkpoints`

| Coluna | Tipo | Descrição |
|---|---|---|
| `source_endpoint` | STRING | Endpoint da API (ex: `/deputados`) |
| `checkpoint_type` | STRING | Tipo de checkpoint (`last_id`, `last_date`) |
| `checkpoint_value` | STRING | Valor do checkpoint (ex: `"500"`, `"2026-04-27"`) |
| `updated_at` | TIMESTAMP | Quando foi atualizado pela última vez |

---

### 💡 Para Leigos

Pense no checkpoint como um **marcador de livro**:
* Você está lendo um livro de 1000 páginas
* Na primeira leitura, lê até página 300 e coloca o marcador
* Na segunda leitura, começa da página 301 (não precisa reler tudo!)
* Full refresh = jogar o marcador fora e começar da página 1 de novo

In [0]:
# ============================================================
# BUSCAR ÚLTIMO CHECKPOINT (ÚLTIMO ID PROCESSADO)
# ============================================================
# Consulta a tabela ops.checkpoints para determinar de onde continuar.
# ============================================================

print("🔖 Buscando último checkpoint...")
print()

# ============================================================
# PASSO 1: Verificar Modo (Full Refresh ou Incremental)
# ============================================================

if full_refresh:
    print("🔄 Modo: FULL REFRESH")
    print("   Checkpoint será ignorado - reprocessando tudo do zero")
    checkpoint_value = None
else:
    print("📈 Modo: INCREMENTAL")
    print("   Buscando último checkpoint salvo...")
    print()
    
    # ============================================================
    # PASSO 2: Consultar Tabela ops.checkpoints
    # ============================================================
    
    query_checkpoint = f"""
    SELECT 
        checkpoint_value,
        updated_at
    FROM {catalog}.{schema_ops}.checkpoints
    WHERE source_endpoint = '/deputados'
      AND checkpoint_type = 'last_id'
    ORDER BY updated_at DESC
    LIMIT 1
    """
    
    try:
        df_checkpoint = spark.sql(query_checkpoint)
        
        # Verifica se encontrou algum checkpoint
        if df_checkpoint.count() > 0:
            checkpoint_row = df_checkpoint.first()
            checkpoint_value = int(checkpoint_row["checkpoint_value"])
            checkpoint_updated_at = checkpoint_row["updated_at"]
            
            print(f"   ✅ Checkpoint encontrado:")
            print(f"      • Último ID processado: {checkpoint_value}")
            print(f"      • Atualizado em:        {checkpoint_updated_at}")
            print(f"      • Processará apenas IDs > {checkpoint_value}")
        else:
            print("   ℹ️  Nenhum checkpoint encontrado (primeira execução?)")
            print("      Processará todos os registros")
            checkpoint_value = None
    
    except Exception as e:
        print(f"   ⚠️  Erro ao buscar checkpoint: {e}")
        print("      Processará todos os registros (modo seguro)")
        checkpoint_value = None

print()
print("="*70)
print(f"📌 Checkpoint definido: {checkpoint_value}")
if checkpoint_value is None:
    print("   → Processará TODOS os deputados")
else:
    print(f"   → Processará apenas deputados com ID > {checkpoint_value}")
print("="*70)
print()

## 🗄️ Bloco 5: Criação da Tabela Bronze

### 📋 Descrição

Cria a tabela `bronze.deputados_raw` se não existir. Esta tabela armazena o **payload JSON completo** sem transformações.

---

### 📊 Schema da Tabela Bronze

| Coluna | Tipo | Descrição |
|---|---|---|
| **`payload`** | STRING | Payload JSON raw completo da API |
| **`record_id`** | STRING | ID único do registro (extraído do JSON) |
| **`record_hash`** | STRING | Hash SHA256 do payload (deduplicação) |
| **`load_id`** | STRING | UUID da execução que ingeriu este registro |
| **`ingestion_timestamp`** | TIMESTAMP | Quando este registro foi ingerido |
| **`ingestion_date`** | DATE | Data de ingestão (partição) |
| **`source_endpoint`** | STRING | Endpoint da API de origem |
| **`env`** | STRING | Ambiente (dev/hml/prd) |

---

### 🎯 Padrão Bronze

1. **Payload JSON raw**: Armazenado como STRING (parse acontece no Silver)
2. **Colunas técnicas**: Auditoria completa (quem, quando, de onde)
3. **Mode APPEND**: Nunca deleta (histórico completo)
4. **Particionado**: Por `ingestion_date` (performance em queries temporais)
5. **Deduplicação**: Via `record_hash` (mesmo registro não é ingerido 2x)

---

### 💡 Para Leigos

A tabela Bronze é como um **arquivo morto** que guarda os documentos originais sem alteração:
* Guardamos o JSON exatamente como veio da API
* Adicionamos etiquetas (colunas técnicas) para rastreabilidade
* Nunca jogamos nada fora (APPEND mode)
* Se precisarmos do original, ele está aqui!

In [0]:
# ============================================================
# CRIAÇÃO DA TABELA BRONZE: deputados_raw
# ============================================================
# Cria tabela Delta se não existir (idempotente).
# ============================================================

print(f"🗄️  Criando tabela Bronze: {table_bronze}...")
print()

# ============================================================
# SQL: CREATE TABLE IF NOT EXISTS
# ============================================================

sql_create_table = f"""
CREATE TABLE IF NOT EXISTS {table_bronze} (
    -- Payload JSON raw completo
    payload STRING COMMENT 'Payload JSON raw completo da API (sem transformações)',
    
    -- ID do registro (extraído do JSON)
    record_id STRING COMMENT 'ID único do registro (deputado.id extraído do JSON)',
    
    -- Hash do payload (para deduplicação e CDC)
    record_hash STRING COMMENT 'Hash SHA256 do payload (identifica mudanças)',
    
    -- Colunas técnicas de auditoria
    load_id STRING COMMENT 'UUID único da execução que ingeriu este registro',
    ingestion_timestamp TIMESTAMP COMMENT 'Timestamp de quando o registro foi ingerido',
    ingestion_date DATE COMMENT 'Data de ingestão (usado para particionamento)',
    source_endpoint STRING COMMENT 'Endpoint da API de origem',
    env STRING COMMENT 'Ambiente de execução (dev/hml/prd)'
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Tabela Bronze - Deputados Raw (payload JSON completo da API Câmara dos Deputados)'
"""

# Executa criação
spark.sql(sql_create_table)

print(f"✅ Tabela '{table_bronze}' criada/validada")
print(f"   • Formato: Delta Lake")
print(f"   • Particionamento: ingestion_date")
print(f"   • Mode: APPEND (histórico completo)")
print()
print("="*70)
print()

## 📥 Bloco 6: Ingestão - Extração da API

### 📋 Descrição

Este bloco chama a função `get_deputados_from_api()` para extrair os dados da API e prepará-los para salvamento.

---
### 🔄 Processo

1. **Chama a API** usando a função definida anteriormente
2. **Recebe lista de payloads JSON** (dicts Python)
3. **Enriquece cada payload** com colunas técnicas:
   * `record_id`: ID do deputado extraído do JSON
   * `record_hash`: Hash SHA256 do payload (deduplicação)
   * `load_id`: UUID desta execução
   * `ingestion_timestamp`: Timestamp atual
   * `ingestion_date`: Data atual
   * `source_endpoint`: `/deputados`
   * `env`: Ambiente (dev/hml/prd)
4. **Converte para DataFrame Spark** para salvar em Delta

---

### 💡 Para Leigos

É como ir ao mercado e colocar as compras no carrinho:
* Pegamos os produtos (dados da API)
* Colocamos etiquetas de preço e validade (colunas técnicas)
* Organizamos no carrinho (DataFrame)
* Próximo passo: passar no caixa (salvar em Bronze)!

In [0]:
# ============================================================
# INGESTÃO: EXTRAIR DADOS DA API
# ============================================================
# Chama API, enriquece com colunas técnicas, prepara DataFrame.
# ============================================================

print("📥 Iniciando ingestão de deputados...")
print()

# Marca timestamp de início
ingest_start_time = time.time()

# ============================================================
# PASSO 1: Chamar API e Extrair Payloads
# ============================================================

deputados_list = get_deputados_from_api(checkpoint_id=checkpoint_value)

num_records_extracted = len(deputados_list)

print(f"✅ {num_records_extracted} deputados extraídos da API")
print()

# ============================================================
# PASSO 2: Verificar Se Há Dados para Processar
# ============================================================

if num_records_extracted == 0:
    print("⚠️  Nenhum registro novo encontrado")
    print("   Nada a processar - finalizando execução")
    print()
    
    # Registra em log que não houve dados novos
    log_bronze_data = [{
        "load_id": load_id,
        "entity_name": entity_name,
        "records_extracted": 0,
        "records_ingested": 0,
        "api_calls_count": 0,
        "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
        "checkpoint_after": str(checkpoint_value) if checkpoint_value else None,
        "execution_time_sec": int(time.time() - ingest_start_time),
        "status": "NO_NEW_DATA",
        "run_date": run_date,
        "env": env,
        "created_at": datetime.now()
    }]
    
    spark.createDataFrame(log_bronze_data).write.mode("append").saveAsTable(
        f"{catalog}.{schema_log}.bronze_ingest_runs"
    )
    
    dbutils.notebook.exit("NO_NEW_DATA")

# ============================================================
# PASSO 3: Enriquecer Payloads com Colunas Técnicas
# ============================================================

print("🔧 Enriquecendo payloads com colunas técnicas...")

enriched_records = []
current_ts = datetime.now()
current_date = current_ts.date()

for deputado_payload in deputados_list:
    
    # Serializa payload como JSON string
    payload_json_str = json.dumps(deputado_payload, ensure_ascii=False)
    
    # Calcula hash SHA256 do payload (para deduplicação)
    payload_hash = hashlib.sha256(payload_json_str.encode('utf-8')).hexdigest()
    
    # Extrai ID do deputado
    record_id = str(deputado_payload.get("id", "unknown"))
    
    # Cria registro enriquecido
    enriched_record = {
        "payload": payload_json_str,
        "record_id": record_id,
        "record_hash": payload_hash,
        "load_id": load_id,
        "ingestion_timestamp": current_ts,
        "ingestion_date": current_date,
        "source_endpoint": "/deputados",
        "env": env
    }
    
    enriched_records.append(enriched_record)

print(f"✅ {len(enriched_records)} registros enriquecidos")
print()

# ============================================================
# PASSO 4: Converter para DataFrame Spark
# ============================================================

print("🔄 Convertendo para DataFrame Spark...")

# Define schema explícito (melhor performance)
from pyspark.sql.types import DateType

schema_bronze = StructType([
    StructField("payload", StringType(), True),
    StructField("record_id", StringType(), True),
    StructField("record_hash", StringType(), True),
    StructField("load_id", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("ingestion_date", DateType(), True),
    StructField("source_endpoint", StringType(), True),
    StructField("env", StringType(), True)
])

df_bronze = spark.createDataFrame(enriched_records, schema=schema_bronze)

print(f"✅ DataFrame criado com {df_bronze.count()} registros")
print()
print("="*70)
print()

## 💾 Bloco 7: Ingestão - Salvar em Bronze

### 📋 Descrição

Salva o DataFrame em `bronze.deputados_raw` usando mode **APPEND**.

---

### 🎯 Mode APPEND

**Por que APPEND e não OVERWRITE?**

* **APPEND**: Adiciona registros novos sem deletar histórico
  * ✅ Mantém auditoria completa (quem ingeriu, quando)
  * ✅ Permite replay (reprocessar períodos específicos)
  * ✅ Suporta CDC (Change Data Capture) via hash comparison
  
* **OVERWRITE**: Deleta tudo e reescreve
  * ❌ Perde histórico de ingestões anteriores
  * ❌ Não permite auditoria temporal
  * ❌ Replay impossível

---

### 🔍 Deduplicação

O campo `record_hash` permite identificar duplicatas:
* Mesmo payload JSON = mesmo hash
* Na camada Silver, faremos deduplicação usando hash
* Bronze mantém TUDO (até duplicatas intencionais)

---

### 📊 Particionamento

Tabela particionada por `ingestion_date`:
* Queries filtradas por data são muito mais rápidas
* Delta Lake otimiza automaticamente (file pruning)
* Exemplo: `WHERE ingestion_date >= '2026-04-01'` só lê arquivos de abril

---

### 💡 Para Leigos

Pense no Bronze como um **diário**:
* Cada dia você escreve novas páginas (APPEND)
* NUNCA arranca páginas antigas (mantém histórico)
* Se quiser ver o que aconteceu em 15/03, vai direto naquele dia (particionamento)
* Se escrever a mesma coisa 2x, fica registrado 2x (deduplicação só no Silver)

In [0]:
# ============================================================
# SALVAR DATAFRAME EM BRONZE (MODE APPEND)
# ============================================================
# Persiste os dados em Delta Lake mantendo histórico completo.
# ============================================================

print(f"💾 Salvando dados em {table_bronze}...")
print()

# Marca tempo de início da escrita
write_start_time = time.time()

try:
    # ============================================================
    # ESCRITA EM DELTA LAKE (APPEND MODE)
    # ============================================================
    
    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ingestion_date") \
        .saveAsTable(table_bronze)
    
    # Calcula tempo de escrita
    write_time_sec = int(time.time() - write_start_time)
    
    print(f"✅ Dados salvos com sucesso!")
    print(f"   • Tabela: {table_bronze}")
    print(f"   • Registros salvos: {num_records_extracted}")
    print(f"   • Tempo de escrita: {write_time_sec}s")
    print(f"   • Mode: APPEND (mantém histórico)")
    print(f"   • Particionamento: ingestion_date")
    print()
    
    ingest_success = True
    ingest_error = None
    
except Exception as e:
    # Se houver erro, captura para log
    write_time_sec = int(time.time() - write_start_time)
    ingest_success = False
    ingest_error = str(e)
    
    print(f"❌ ERRO ao salvar dados: {e}")
    print()
    raise

finally:
    # Calcula tempo total de ingestão
    ingest_total_time_sec = int(time.time() - ingest_start_time)
    
    print("="*70)
    print(f"⏱️  Tempo total de ingestão: {ingest_total_time_sec}s")
    print("="*70)
    print()

## 🔖 Bloco 8: Atualização do Checkpoint

### 📋 Descrição

Atualiza a tabela `ops.checkpoints` com o **novo último ID processado**.

---

### 🎯 Por Que Atualizar Checkpoint?

O checkpoint marca **até onde o pipeline já processou**:
* Permite execuções incrementais (processar apenas novos registros)
* Evita reprocessamento desnecessário
* Garante exatamente-once semantics (com idempotência)

---

### 🔄 Como Funciona

1. **Calcula novo checkpoint**: Maior ID dos registros que acabamos de ingerir
2. **MERGE em ops.checkpoints**: Atualiza se já existe, insere se não existe
3. **Próxima execução**: Começará do ID seguinte ao checkpoint

**Exemplo:**
* **Agora**: Ingerimos deputados ID 1-500 → Checkpoint = 500
* **Próxima execução**: Processará apenas ID > 500

---

### 🔍 MERGE vs INSERT

**Por que MERGE e não INSERT?**

Se fizermos apenas INSERT:
* Primeira execução: INSERT checkpoint 500 ✅
* Segunda execução: INSERT checkpoint 520 ✅
* Resultado: 2 linhas (qual é o válido?) ❌

Com MERGE:
* WHEN MATCHED: Atualiza linha existente
* WHEN NOT MATCHED: Insere nova linha
* Resultado: Sempre 1 linha por (endpoint + checkpoint_type) ✅

---

### 💡 Para Leigos

O checkpoint é como o **marcador de páginas lidas**:
* Você lê até página 500 → Coloca marcador na 500
* Próxima vez: Começa da página 501
* Se atualizar o marcador errado, vai reler páginas (reprocessamento inútil)
* Se esquecer de atualizar, vai reler tudo (full refresh acidental)

In [0]:
# ============================================================
# ATUALIZAR CHECKPOINT (ÚLTIMO ID PROCESSADO)
# ============================================================
# Salva o maior ID ingerido para próximas execuções incrementais.
# ============================================================

print("🔖 Atualizando checkpoint...")
print()

# ============================================================
# PASSO 1: Calcular Novo Checkpoint (Maior ID Ingerido)
# ============================================================

# Extrai o maior record_id do DataFrame que acabamos de salvar
max_id_row = df_bronze.selectExpr("CAST(record_id AS INT) as id_int") \
    .agg({"id_int": "max"}) \
    .first()

new_checkpoint_value = max_id_row[0]  # Maior ID

print(f"📊 Checkpoint calculado:")
print(f"   • Checkpoint anterior: {checkpoint_value}")
print(f"   • Novo checkpoint:     {new_checkpoint_value}")
print()

if new_checkpoint_value is None:
    print("⚠️  Nenhum ID válido encontrado - checkpoint não será atualizado")
    print()
else:
    # ============================================================
    # PASSO 2: MERGE em ops.checkpoints
    # ============================================================
    # Atualiza linha existente ou insere nova.
    
    sql_merge_checkpoint = f"""
    MERGE INTO {catalog}.{schema_ops}.checkpoints AS target
    USING (
        SELECT 
            '/deputados' AS source_endpoint,
            'last_id' AS checkpoint_type,
            '{new_checkpoint_value}' AS checkpoint_value,
            current_timestamp() AS updated_at
    ) AS source
    ON target.source_endpoint = source.source_endpoint
       AND target.checkpoint_type = source.checkpoint_type
    WHEN MATCHED THEN 
        UPDATE SET 
            target.checkpoint_value = source.checkpoint_value,
            target.updated_at = source.updated_at
    WHEN NOT MATCHED THEN
        INSERT (source_endpoint, checkpoint_type, checkpoint_value, updated_at)
        VALUES (source.source_endpoint, source.checkpoint_type, source.checkpoint_value, source.updated_at)
    """
    
    spark.sql(sql_merge_checkpoint)
    
    print(f"✅ Checkpoint atualizado com sucesso!")
    print(f"   • Endpoint:       /deputados")
    print(f"   • Tipo:           last_id")
    print(f"   • Valor:          {new_checkpoint_value}")
    print(f"   • Próxima exec:   Processará IDs > {new_checkpoint_value}")
    print()

print("="*70)
print()

## 📝 Bloco 9: Logging Detalhado

### 📋 Descrição

Registra métricas detalhadas desta ingestão em `log.bronze_ingest_runs`.

---

### 📊 Tabela: `log.bronze_ingest_runs`

Armazena **1 linha por entidade ingerida em cada execução**.

| Coluna | Tipo | Descrição |
|---|---|---|
| `load_id` | STRING | UUID da execução |
| `entity_name` | STRING | Nome da entidade (`deputados`) |
| `records_extracted` | INT | Quantos registros a API retornou |
| `records_ingested` | INT | Quantos registros foram salvos |
| `api_calls_count` | INT | Número de chamadas HTTP feitas |
| `checkpoint_before` | STRING | Checkpoint antes desta execução |
| `checkpoint_after` | STRING | Checkpoint depois desta execução |
| `execution_time_sec` | INT | Tempo total de execução (segundos) |
| `status` | STRING | `SUCCESS`, `FAILED`, `NO_NEW_DATA` |
| `error_message` | STRING | Mensagem de erro (se `FAILED`) |
| `run_date` | DATE | Data de referência da execução |
| `env` | STRING | Ambiente (dev/hml/prd) |
| `created_at` | TIMESTAMP | Quando este log foi criado |

---

### 🎯 Para Que Serve Este Log?

**Troubleshooting:**
* "Por que a ingestão de hoje foi mais lenta?"
* "Quantos registros foram ingeridos na semana passada?"
* "Houve algum erro na execução de ontem?"

**Auditoria:**
* "Qual load_id ingeriu o deputado ID 12345?"
* "Quantas chamadas à API fizemos em abril?"

**Alertas:**
* Se `records_ingested = 0` por vários dias → Possível problema na API
* Se `execution_time_sec` > threshold → Alerta de lentidão

---

### 💡 Para Leigos

É como o **livro de bordo de um navio**:
* Registra cada viagem (execução)
* Anota distância percorrida (registros ingeridos)
* Marca tempo de viagem (execution_time_sec)
* Anota incidentes (error_message)
* Se algo der errado, consulta o livro para investigar!

In [0]:
# ============================================================
# REGISTRAR LOG DETALHADO EM log.bronze_ingest_runs
# ============================================================
# Persiste métricas desta ingestão para auditoria e troubleshooting.
# ============================================================

print("📝 Registrando log detalhado...")
print()

# ============================================================
# PASSO 1: Preparar Dados do Log
# ============================================================

# Conta quantas chamadas HTTP foram feitas (consulta ops.ingestion_requests)
query_api_calls = f"""
SELECT COUNT(*) as api_calls
FROM {catalog}.{schema_ops}.ingestion_requests
WHERE load_id = '{load_id}'
  AND entity_name = '{entity_name}'
"""

api_calls_count = spark.sql(query_api_calls).first()[0]

# Prepara dados do log
log_bronze_data = [{
    "load_id": load_id,
    "entity_name": entity_name,
    "records_extracted": num_records_extracted,
    "records_ingested": num_records_extracted,  # Mesmo valor (Bronze não rejeita)
    "api_calls_count": api_calls_count,
    "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
    "checkpoint_after": str(new_checkpoint_value) if new_checkpoint_value else None,
    "execution_time_sec": ingest_total_time_sec,
    "status": "SUCCESS" if ingest_success else "FAILED",
    "error_message": ingest_error,
    "run_date": run_date,
    "env": env,
    "created_at": datetime.now()
}]

# ============================================================
# PASSO 2: Salvar Log (APPEND)
# ============================================================

df_log = spark.createDataFrame(log_bronze_data)

df_log.write.mode("append").saveAsTable(
    f"{catalog}.{schema_log}.bronze_ingest_runs"
)

print(f"✅ Log registrado em {catalog}.{schema_log}.bronze_ingest_runs")
print(f"   • Load ID:              {load_id}")
print(f"   • Entidade:             {entity_name}")
print(f"   • Registros ingeridos:  {num_records_extracted}")
print(f"   • Chamadas à API:       {api_calls_count}")
print(f"   • Tempo de execução:    {ingest_total_time_sec}s")
print(f"   • Status:               SUCCESS")
print()
print("="*70)
print()

## ✅ Bloco 10: Validação dos Dados Ingeridos

### 📋 Descrição

Valida que os dados foram salvos corretamente em `bronze.deputados_raw`.

---

### 🔍 Validações Realizadas

1. **Contagem Total**: Quantos registros existem na tabela?
2. **Contagem deste Load**: Quantos registros foram ingeridos nesta execução?
3. **Amostra de Dados**: Exibe 5 primeiros registros (visual check)
4. **Schema Validation**: Confirma que todas as colunas existem

---

### 🎯 O Que Procurar

**✅ Sinais de Sucesso:**
* Contagem > 0 (há dados)
* Registros deste load_id aparecem
* Payload não está NULL
* ingestion_date = hoje (ou run_date)
* record_hash tem valores (64 caracteres hex)

**❌ Sinais de Problema:**
* Contagem = 0 (nada foi salvo?)
* Payload NULL (erro no enriquecimento?)
* ingestion_date errada (timezone?)
* Duplicatas inesperadas (hash repetido)

---

### 💡 Para Leigos

É como conferir as compras quando chega do mercado:
* Contou quantas sacolas vieram? (contagem)
* Abriu uma sacola para ver se está tudo certo? (amostra)
* Conferiu se não falta nada? (schema validation)
* Verificou validade dos produtos? (colunas técnicas)

In [0]:
# ============================================================
# VALIDAÇÃO DOS DADOS INGERIDOS
# ============================================================
# Verifica que os dados foram salvos corretamente em Bronze.
# ============================================================

print("✅ Validando dados ingeridos...")
print()
print("="*70)

# ============================================================
# VALIDAÇÃO 1: Contagem Total de Registros
# ============================================================

query_count_total = f"""
SELECT COUNT(*) as total_records
FROM {table_bronze}
"""

total_records = spark.sql(query_count_total).first()[0]

print(f"📊 VALIDAÇÃO 1: Contagem Total")
print(f"   • Total de registros na tabela: {total_records:,}")
print()

# ============================================================
# VALIDAÇÃO 2: Contagem Deste Load ID
# ============================================================

query_count_this_load = f"""
SELECT COUNT(*) as records_this_load
FROM {table_bronze}
WHERE load_id = '{load_id}'
"""

records_this_load = spark.sql(query_count_this_load).first()[0]

print(f"📊 VALIDAÇÃO 2: Contagem Deste Load")
print(f"   • Registros ingeridos nesta execução: {records_this_load:,}")
print(f"   • Esperado: {num_records_extracted:,}")

if records_this_load == num_records_extracted:
    print(f"   ✅ Contagem corresponde!")
else:
    print(f"   ⚠️  ATENÇÃO: Divergência na contagem!")

print()

# ============================================================
# VALIDAÇÃO 3: Amostra de Dados (5 primeiros registros)
# ============================================================

print(f"📊 VALIDAÇÃO 3: Amostra de Dados (5 primeiros deste load)")
print()

query_sample = f"""
SELECT 
    record_id,
    LEFT(record_hash, 12) as hash_preview,
    ingestion_timestamp,
    ingestion_date,
    env,
    LEFT(payload, 100) as payload_preview
FROM {table_bronze}
WHERE load_id = '{load_id}'
ORDER BY ingestion_timestamp
LIMIT 5
"""

df_sample = spark.sql(query_sample)

display(df_sample)

print()

# ============================================================
# VALIDAÇÃO 4: Estatísticas por Partição
# ============================================================

print(f"📊 VALIDAÇÃO 4: Estatísticas por Partição (ingestion_date)")
print()

query_partitions = f"""
SELECT 
    ingestion_date,
    COUNT(*) as records_count,
    COUNT(DISTINCT load_id) as distinct_loads
FROM {table_bronze}
GROUP BY ingestion_date
ORDER BY ingestion_date DESC
LIMIT 10
"""

df_partitions = spark.sql(query_partitions)

display(df_partitions)

print()
print("="*70)
print()
print("✅ VALIDAÇÃO CONCLUÍDA!")
print()
print(f"🎯 Resumo Final:")
print(f"   • Tabela:                {table_bronze}")
print(f"   • Total de registros:    {total_records:,}")
print(f"   • Novos nesta execução:  {records_this_load:,}")
print(f"   • Load ID:               {load_id}")
print(f"   • Checkpoint atualizado: {new_checkpoint_value}")
print(f"   • Status:                SUCCESS ✅")
print()
print("="*70)
print()
print("🎉 Notebook 01_bronze_deputados concluído com sucesso!")